## HuggingFace Notebook 🤗
Loading code to view our speech corpus datasets on huggingface! 

In [ ]:
from datasets import load_dataset
from IPython.display import Audio, display
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

from scripts.run_evaluation import run_eval_corpus

In [ ]:
from huggingface_hub import login
login()  # enter your HuggingFace token when prompted

In [ ]:
ds = load_dataset("multispeak/hiring-accent-corpus", split="train")
ds

In [ ]:
# Filter to just speaker_38 for re-eval
ds_rohan = ds.filter(lambda r: r["name"] == "speaker_12")
print(f"{len(ds_rohan)} samples for speaker_38")

### Listen to examples

In [ ]:
def play_sample(sample):
    print(f"Speaker : {sample['name']}")
    print(f"Category: {sample['category']}")
    print(f"Accent  : {sample['accent_nationality_origin']}")
    print(f"Quality : {sample['audio_quality_rating']} — {sample['quality_notes']}")
    audio = sample["audio"]
    plt.figure(figsize=(10, 2))
    plt.plot(audio["array"])
    plt.title(f"{sample['name']} — {sample['category']}")
    plt.tight_layout()
    plt.show()
    display(Audio(audio["array"], rate=audio["sampling_rate"]))

In [ ]:
# Filter by category and accent, then play
CATEGORY = "personal-introduction"   # change as needed
ACCENT   = "Nigerian"            # change as needed

subset = ds.filter(lambda r: r["category"] == CATEGORY and r["accent_nationality_origin"] == ACCENT)
print(f"{len(subset)} samples found")
play_sample(subset[0])

### Evals!


In [ ]:
PROMPT_FILES = {
    "critical": "../audio_samples/elevenlabs_dataset4/prompts_two_part_rating_critical.csv",
    "ideal":    "../audio_samples/elevenlabs_dataset4/prompts_two_part_rating_ideal.csv",
    "native":   "../audio_samples/elevenlabs_dataset4/prompts_two_part_rating_native.csv",
}

run_eval_corpus(
    model_name   = "gemini-2.5-flash",
    hf_dataset   = ds_rohan,
    prompt_files = PROMPT_FILES,
    output_path  = "../results/hiring_corpus/gemini-2.5-flash_hiring_corpus.csv",
)


In [ ]:
# number of hours of speech
total_seconds = sum([len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"] for sample in ds])
total_hours = total_seconds / 3600
print(f"Total hours of speech: {total_hours:.2f} hours")

In [ ]:
# Retry the 5 failed rows (API errors marked as FAILED)
RETRY_SPEAKERS = ["speaker_13", "speaker_02", "speaker_04", "speaker_05", "speaker_06"]
ds_retry = ds.filter(lambda r: r["name"] in RETRY_SPEAKERS)
print(f"{len(ds_retry)} samples to retry")

run_eval_corpus(
    model_name   = "gemini-2.5-flash",
    hf_dataset   = ds_retry,
    prompt_files = PROMPT_FILES,
    output_path  = "../results/hiring_corpus/gemini-2.5-flash_hiring_corpus.csv",
)
